# RAG Findings Visualization

This notebook builds publication-ready tables and figures from:
- `results/rag_vs_llm_only_e5_multiseed_summary.json` (fixed k baseline)
- `results/rag_vs_llm_only_e5_tuned_multiseed_summary.json` (tuned language-specific k)
- `results/rag_k_sweep_multiseed_summary.json` (k-sweep curves, optional)

In [2]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

# Robust results directory resolution
candidates = [
    Path("results"),
    Path("../results"),
    Path.cwd() / "results",
]
RESULTS_DIR = next((p for p in candidates if p.exists()), Path("results"))

tuned_path = RESULTS_DIR / "rag_vs_llm_only_e5_tuned_multiseed_summary.json"
fixed_path = RESULTS_DIR / "rag_vs_llm_only_e5_multiseed_summary.json"
k_sweep_path = RESULTS_DIR / "rag_k_sweep_multiseed_summary.json"

if not tuned_path.exists():
    raise FileNotFoundError("Missing tuned summary: {}".format(tuned_path))
if not fixed_path.exists():
    raise FileNotFoundError("Missing fixed summary: {}".format(fixed_path))

with open(tuned_path, "r", encoding="utf-8") as f:
    tuned = json.load(f)
with open(fixed_path, "r", encoding="utf-8") as f:
    fixed = json.load(f)

langs = ["swa", "yor", "kin"]

rows = []
for lang in langs:
    t = tuned["by_language"][lang]
    rows.append({
        "language": lang.upper(),
        "rag_mean": t["rag_correct_rate"]["mean"],
        "rag_std": t["rag_correct_rate"]["std"],
        "llm_mean": t["llm_only_correct_rate"]["mean"],
        "llm_std": t["llm_only_correct_rate"]["std"],
        "delta_mean": t["delta_rag_minus_llm_only_correct_rate"]["mean"],
        "delta_std": t["delta_rag_minus_llm_only_correct_rate"]["std"],
        "sim_mean": t["rag_mean_similarity"]["mean"],
        "sim_std": t["rag_mean_similarity"]["std"],
    })

df_tuned = pd.DataFrame(rows).set_index("language")

rows_cmp = []
for lang in langs:
    rows_cmp.append({
        "language": lang.upper(),
        "fixed_delta_mean": fixed["by_language"][lang]["delta_rag_minus_llm_only_correct_rate"]["mean"],
        "fixed_delta_std": fixed["by_language"][lang]["delta_rag_minus_llm_only_correct_rate"]["std"],
        "tuned_delta_mean": tuned["by_language"][lang]["delta_rag_minus_llm_only_correct_rate"]["mean"],
        "tuned_delta_std": tuned["by_language"][lang]["delta_rag_minus_llm_only_correct_rate"]["std"],
    })

df_cmp = pd.DataFrame(rows_cmp).set_index("language")

display(df_tuned)
display(df_cmp)

print("Loaded from:", RESULTS_DIR.resolve())

FileNotFoundError: Missing tuned summary: results\rag_vs_llm_only_e5_tuned_multiseed_summary.json

In [ ]:
# Table A: Tuned multi-seed results as percentages
summary_table = pd.DataFrame({
    "RAG Correct Rate": ["{:.1f}% ± {:.1f}%".format(v["rag_mean"] * 100, v["rag_std"] * 100) for _, v in df_tuned.iterrows()],
    "LLM-only Correct Rate": ["{:.1f}% ± {:.1f}%".format(v["llm_mean"] * 100, v["llm_std"] * 100) for _, v in df_tuned.iterrows()],
    "Delta (RAG-LLM)": ["{:+.1f}% ± {:.1f}%".format(v["delta_mean"] * 100, v["delta_std"] * 100) for _, v in df_tuned.iterrows()],
    "Retriever Similarity": ["{:.1f}% ± {:.2f}%".format(v["sim_mean"] * 100, v["sim_std"] * 100) for _, v in df_tuned.iterrows()],
}, index=df_tuned.index)

summary_table

In [ ]:
# Plot 1: RAG vs LLM-only correctness (tuned)
x = np.arange(len(df_tuned.index))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width / 2, df_tuned["rag_mean"], yerr=df_tuned["rag_std"], width=width, capsize=4, label="RAG")
ax.bar(x + width / 2, df_tuned["llm_mean"], yerr=df_tuned["llm_std"], width=width, capsize=4, label="LLM-only")

ax.set_xticks(x)
ax.set_xticklabels(df_tuned.index)
ax.set_title("Correct Rate: RAG vs LLM-only (Tuned k)")
ax.set_ylabel("Correct Rate")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot 2: Delta (RAG - LLM-only) with uncertainty
x = np.arange(len(df_tuned.index))

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x, df_tuned["delta_mean"], yerr=df_tuned["delta_std"], capsize=4)
ax.axhline(0, color="black", linewidth=1, linestyle="--")

ax.set_xticks(x)
ax.set_xticklabels(df_tuned.index)
ax.set_title("Delta Correct Rate (RAG - LLM-only), Tuned k")
ax.set_ylabel("Delta Correct Rate")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))

for i, v in enumerate(df_tuned["delta_mean"].values):
    ax.text(i, v + 0.003, "{:.1f}%".format(v * 100), ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Plot 3: Before vs after tuning (delta comparison)
x = np.arange(len(df_cmp.index))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width / 2, df_cmp["fixed_delta_mean"], yerr=df_cmp["fixed_delta_std"], width=width, capsize=4, label="Fixed k")
ax.bar(x + width / 2, df_cmp["tuned_delta_mean"], yerr=df_cmp["tuned_delta_std"], width=width, capsize=4, label="Tuned k")

ax.axhline(0, color="black", linewidth=1, linestyle="--")
ax.set_xticks(x)
ax.set_xticklabels(df_cmp.index)
ax.set_title("Delta Improvement: Fixed k vs Tuned k")
ax.set_ylabel("Delta Correct Rate (RAG - LLM-only)")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot 4: Retriever similarity stability (tuned)
x = np.arange(len(df_tuned.index))

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x, df_tuned["sim_mean"], yerr=df_tuned["sim_std"], capsize=4)

ax.set_xticks(x)
ax.set_xticklabels(df_tuned.index)
ax.set_title("Retriever Mean Similarity by Language (Tuned k)")
ax.set_ylabel("Similarity")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
plt.tight_layout()
plt.show()

In [ ]:
# Plot 5 (optional): k-sweep delta curves
if not k_sweep_path.exists():
    print("Skipping: {} not found".format(k_sweep_path))
else:
    with open(k_sweep_path, "r", encoding="utf-8") as f:
        k_sweep = json.load(f)

    fig, ax = plt.subplots(figsize=(10, 5))
    k_values = [str(k) for k in k_sweep["metadata"]["k_values"]]

    for lang in langs:
        y = [k_sweep["by_language"][lang]["rag_by_k"][k]["delta_rag_minus_llm_only_correct_rate"]["mean"] for k in k_values]
        yerr = [k_sweep["by_language"][lang]["rag_by_k"][k]["delta_rag_minus_llm_only_correct_rate"]["std"] for k in k_values]
        ax.errorbar([int(k) for k in k_values], y, yerr=yerr, marker="o", capsize=4, label=lang.upper())

    ax.axhline(0, color="black", linewidth=1, linestyle="--")
    ax.set_title("k-sweep: Delta Correct Rate (RAG - LLM-only)")
    ax.set_xlabel("k")
    ax.set_ylabel("Delta Correct Rate")
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.legend()
    plt.tight_layout()
    plt.show()